# Capstone walkthrough — ingest → RAG → agent → gateway → serve → eval

One system from weeks 5–10. This notebook builds it up with `llmlab.FakeLLM` (offline, free),
then shows the FastAPI surface and the eval gate. Swap `FakeLLM` for `get_client("anthropic")`
and it's the real thing (that's what `make live` does).

Implement `src/ragplatform/` until `make test` is green, then run this top to bottom.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd() / "solution"))
sys.path.insert(0, str(pathlib.Path.cwd() / "fixtures"))
sys.path.insert(0, str(pathlib.Path.cwd() / "tests"))
from ragplatform import RagPlatform, EvalGate, create_app
from kb import DOCS, EVAL_CASES
from helpers import kb_fake
from fastapi.testclient import TestClient
print("docs:", list(DOCS))

## 1 — Build the platform (ingest + index + RAG + agent + gateway)

In [ ]:
plat = RagPlatform(DOCS, kb_fake())
print(plat.ask("how long do I have to ask for a refund?"))
print(plat.ask("do you sell hardware?"))              # out of scope -> refusal
print("gateway stats after 2 asks:", plat.gateway.stats)
plat.ask("how long do I have to ask for a refund?")   # repeat -> cache hit
print("gateway stats after repeat :", plat.gateway.stats)

## 2 — Serve it (API key + rate limit)

In [ ]:
client = TestClient(create_app(plat, api_key="secret", rate=100, burst=100))
print("no key :", client.post("/ask", json={"question": "refund window?"}).status_code)
r = client.post("/ask", json={"question": "what is the minimum password length?"},
                headers={"x-api-key": "secret"})
print("with key:", r.status_code, r.json())

## 3 — Eval gate: does a change regress quality?

In [ ]:
from llmlab import FakeLLM
judge_ok = FakeLLM(lambda m, **k: ("tool", "emit", {"score": 0.85, "reasoning": "grounded"}))
gate = EvalGate(judge_ok)
baseline = gate.check(lambda q: plat.ask(q), EVAL_CASES, baseline=None)
print("baseline mean:", round(baseline.mean, 2), "-> this run becomes the baseline")

# simulate a worse deploy: the judge now scores answers low
judge_bad = FakeLLM(lambda m, **k: ("tool", "emit", {"score": 0.3, "reasoning": "vague"}))
res = EvalGate(judge_bad).check(lambda q: plat.ask(q), EVAL_CASES, baseline=baseline.per_case)
print("candidate ok?", res.ok, " mean", round(res.mean, 2), "vs", round(res.baseline_mean, 2))

## 4 — The trace

Every layer wrote a span (`gateway.chat`, `rag.answer`, `agent.run`). In production these go
to your tracing backend (Day 27 / Day 34); here, print the last one.

In [ ]:
from llmlab import span, render
with span("demo.request") as root:
    plat.ask("where do I turn on 2FA?", agentic=False)
print(render(root))

## 5 — Ship it for real

```bash
make live      # RagPlatform(DOCS, get_client("anthropic")) + a real judge, ~$0.15
```

From here: put `create_app` behind the Day 30 Lambda/API-Gateway deploy, wrap the pipeline in
the Day 33 eval-gated CI, and point the Day 34 `RefreshTrigger` at its trace stream. That's
the whole course, running.